In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy
import skimage
from scipy.ndimage import rotate
from scipy.signal import argrelmin
from sklearn.cluster import DBSCAN


def bump(x):
    return np.where(np.abs(x) < 1, np.exp(1 / (x ** 2 - 1) + 1) ** 0.2, 0)


def argrelmin_rotated(arr, angle, axis=0, order=1,
                      reshape=True, interp_order=1,
                      mode='constant', cval=np.inf):
    """
    Find relative minima in a 2D array along lines at a given angle
    by rotating the data, using scipy.signal.argrelmin, then mapping
    those minima back to the original array coordinates.

    Parameters
    ----------
    arr : array_like, shape (M, N)
        Your input 2D signal.
    angle : float
        Angle in degrees, measured counter‑clockwise from the +x axis (columns).
    axis : {0, 1}, optional
        Like scipy.signal.argrelmin: axis=0 finds minima down columns
        (i.e. along lines at `angle` in the original), axis=1 along rows
        (lines at `angle + 90°`).
    order : int, optional
        How many neighbors on each side to demand being larger (default 1).
    reshape : bool, optional
        Passed to scipy.ndimage.rotate.  If True, the rotated image
        is large enough to contain the entire original.
    interp_order : int, optional
        The spline‐interpolation order for the rotation (0=nearest, 1=bilinear, …).
        Nearest (0) preserves your exact pixel values.
    mode, cval : optional
        Passed to scipy.ndimage.rotate for areas outside the original image.
        We default to constant `+∞` so no bogus minima appear on the padding.

    Returns
    -------
    minima_mask : ndarray of bool, shape (M, N)
        True in the original array where a strict relative minimum
        was found along the specified angled direction.
    """
    arr = np.asarray(arr)
    M, N = arr.shape

    # rotate by –angle so that lines at +angle become vertical (axis=0)
    arr_rot = rotate(arr, -angle,
                     reshape=reshape,
                     order=interp_order,
                     mode=mode,
                     cval=cval)
    M_rot, N_rot = arr_rot.shape

    # find relative minima in the rotated frame
    rows_rot, cols_rot = argrelmin(arr_rot, axis=axis, order=order)

    # compute centers of original and rotated images
    center_orig = np.array([(M - 1) / 2.0, (N - 1) / 2.0])
    center_rot = np.array([(M_rot - 1) / 2.0, (N_rot - 1) / 2.0])

    # vectorized inverse‐rotation of all minima positions
    theta = np.deg2rad(angle)
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    # offsets from rotated‐center
    dr = rows_rot - center_rot[0]
    dc = cols_rot - center_rot[1]

    # inverse rotation back to original coords
    #  [ x_orig ]   [  cos  sin ] [ dc ]   + center_orig[1]
    #  [ y_orig ] = [ -sin  cos ] [ dr ]   + center_orig[0]
    col_orig = cos_t * dc + sin_t * dr + center_orig[1]
    row_orig = -sin_t * dc + cos_t * dr + center_orig[0]

    # round to nearest pixel and filter valid indices
    row_idx = np.round(row_orig).astype(int)
    col_idx = np.round(col_orig).astype(int)
    valid = (
            (row_idx >= 0) & (row_idx < M) &
            (col_idx >= 0) & (col_idx < N)
    )
    return row_idx[valid], col_idx[valid]


def extract_theory_df(I, r_v, theta_v, sigma_small=25, sigma_large=150, angle=60, poly_order=12, spline_lambda=1e-7,
                      e_scale=1):
    """
    Process intensity I and coordinate arrays r_v, theta_v.
    Returns a downsampled DataFrame with columns ['pr','E','theta','group_delay','signal'].
    """
    # FFT and initial signal
    fft = np.fft.fft(I, axis=1)
    tot = np.abs(fft[:, 0])
    recovered = skimage.filters.difference_of_gaussians(I, sigma_small, sigma_large, mode='wrap')

    # Rotated minima detection
    args_r = argrelmin_rotated(recovered, angle=angle, axis=1, order=1)
    r_vals, theta_vals = r_v[args_r[0]], theta_v[args_r[1]]

    # Filter by radial range
    mask = (r_vals > 0.13) & (r_vals < 0.3) & (theta_vals > 0) & (theta_vals < 2)
    r_min, theta_min = r_vals[mask], theta_vals[mask]

    # Cluster and select the largest cluster
    labels = DBSCAN(eps=0.1, min_samples=1).fit_predict(np.vstack((r_min * 15, theta_min)).T)
    sel = labels == np.argmax(np.bincount(labels))
    r_min, theta_min = r_min[sel], theta_min[sel]

    # Aggregate by theta
    df_pts = pd.DataFrame({'r': r_min, 'theta': theta_min})
    df_pts = df_pts.groupby('theta', as_index=False)['r'].min()
    r_min, theta_min = df_pts['r'].values, df_pts['theta'].values

    # Contrast weights
    crs = np.max(I, axis=1) - np.min(I, axis=1)
    idxs = np.intersect1d(theta_v, theta_min, return_indices=True)[1]
    cr = crs[idxs]

    # Polynomial smoothing of theta(E)
    E_min = r_min ** 2 / 2
    coeffs = np.polyfit(E_min, theta_min, poly_order, w=cr)
    r_sm = np.linspace(r_min.min(), r_min.max(), len(r_min))
    E = r_sm ** 2 / 2
    theta_sm = np.poly1d(coeffs)(E)
    group_delay = np.gradient(theta_sm, E * e_scale) * 2

    # Smooth the tot signal
    tot_sm = scipy.interpolate.make_smoothing_spline(r_v ** 2 / 2, tot, lam=spline_lambda)(E)

    theory_df = pd.DataFrame({
        'pr': r_min,
        'E': E * e_scale,
        'theta': theta_sm,
        'group_delay': group_delay,
        'signal': tot_sm / np.max(tot_sm)
    })
    return theory_df

In [ ]:
# Load and filter the theory data
theory_file = r"C:\DATA\4ranges-volume-momspe-2d-sw.dat"
data = []
with open(theory_file) as f:
    for line in f:
        parts = line.strip().split("  ")
        if len(parts) == 6:
            data.append([float(x) for x in parts])
data = np.asarray(data)
data = data[data[:, 0] < 0.35]


def fit_fn(theta, theta0, a, b):
    """Cosine fitting function for angular data."""
    return a * np.cos((theta - theta0) * 2) + b


# Extract data components
pr = data[:, 0]  # Momentum transfer values
theta = data[:, 1] % (2 * np.pi) - np.pi  # Angle values normalized to [-π, π]
intensity = data[:, -1]  # Intensity values

# Reshape to 2D grid based on unique values
grid_size = (len(set(pr)), len(set(theta)))
pr = pr.reshape(grid_size)
theta = theta.reshape(grid_size)
intensity = intensity.reshape(grid_size)
r_values = np.asarray(sorted(np.unique(pr)))
theta_values = np.asarray(sorted(np.unique(theta)))

# Interpolate onto regular grid with 1024 points
RESOLUTION = 1024
grid_indices = (np.linspace(0, len(r_values), RESOLUTION),
                np.linspace(0, len(theta_values), RESOLUTION))
rr, tt = np.meshgrid(*grid_indices)
intensity = scipy.ndimage.map_coordinates(intensity, [rr.flatten(), tt.flatten()])

# Create physical coordinate meshgrid
r_values = np.linspace(min(r_values), max(r_values), RESOLUTION)
theta_values = np.linspace(min(theta_values), max(theta_values), RESOLUTION)
pr, theta = np.meshgrid(r_values, theta_values)
intensity = intensity.reshape(RESOLUTION, RESOLUTION).T

I = intensity * bump((pr.T - 0.2) / 0.1)

In [ ]:
f = px.imshow(I, x=theta_values, y=r_values, origin='lower', aspect='auto').update_layout(
        title="Interlocking Error Bars",
        yaxis_title="Momentum (pr)",
        xaxis_title="Angle (theta)"
)

f2 = go.Figure()

theory_dfs = []

scale = 0.874 ** -2
scale = 1

for angle in np.arange(18, 81, 3):
    theory_df = extract_theory_df(I, r_values, theta_values, angle=angle, e_scale=scale)
    f.add_scatter(x=theory_df['theta'], y=theory_df['pr'], mode='lines', name=f'{angle}°')
    f2.add_scatter(x=theory_df['E'], y=theory_df['group_delay'], mode='lines', name=f'{angle}°')
    theory_dfs.append(theory_df)

for hs in np.arange(100, 200, 4):
    theory_df = extract_theory_df(I, r_values, theta_values, sigma_large=hs, e_scale=scale)
    f.add_scatter(x=theory_df['theta'], y=theory_df['pr'], mode='lines', name=f'hs={hs}')
    f2.add_scatter(x=theory_df['E'], y=theory_df['group_delay'], mode='lines', name=f'hs={hs}')
    theory_dfs.append(theory_df)

for ls in np.arange(10, 50, 2):
    theory_df = extract_theory_df(I, r_values, theta_values, sigma_small=ls, e_scale=scale)
    f.add_scatter(x=theory_df['theta'], y=theory_df['pr'], mode='lines', name=f'ls={ls}')
    f2.add_scatter(x=theory_df['E'], y=theory_df['group_delay'], mode='lines', name=f'ls={ls}')
    theory_dfs.append(theory_df)

# Show the figure
f.show()
f2.update_layout(
        title="Group Delay vs Energy (Theory)",
        yaxis_title="Group Delay",
        xaxis_title="Energy (E)",
        yaxis_range=[0, 250],
        xaxis_range=[0.015, 0.035],
).show()

In [ ]:
# Load experimental data
fname_exp = r"J:\ctgroup\Edward\DATA\VMI\20220613\xe005_e_calibrated.h5"
base_data = pd.read_hdf(fname_exp, key="data")

# Filter and symmetrize
data = (base_data
        .query("sqrt(px**2+py**2+pz**2)<0.4")
        .query("abs(py)<0.5")
        .query("pz>0")
        .reset_index(drop=True))
data_sym = data.copy()
data_sym[['px', 'py', 'pz']] *= -1
data = pd.concat([data, data_sym], ignore_index=True)

# Compute spherical coordinates
data['pr'] = np.linalg.norm(data[['px', 'py', 'pz']].values, axis=1)
data['phi'] = np.arctan2(data['px'], data['pz'])
data['theta'] = np.arccos(data['px'] / data['pr'])
data['E'] = data['pr'] ** 2 / 2

# Build 2D histogram
RES = 1024
h2d, re_edges, pe_edges = np.histogram2d(data['pr'], data['phi'], bins=RES)
r_vals = 0.5 * (re_edges[:-1] + re_edges[1:])
theta_vals = 0.5 * (pe_edges[:-1] + pe_edges[1:])
I_exp = (h2d.T * bump((r_vals - 0.25) / 0.1)).T


In [ ]:
#create same plots as theory
f_exp = px.imshow(I_exp, x=theta_vals, y=r_vals, origin='lower', aspect='auto').update_layout(
        title="Experimental Interlocking Error Bars",
        yaxis_title="Momentum (pr)",
        xaxis_title="Angle (phi)"
)
f2_exp = go.Figure()
exp_dfs = []

scale = 0.874 ** 2
# scale=1
for angle in np.arange(0, 90, 3):
    df = extract_theory_df(I_exp, r_vals, theta_vals, angle=angle, e_scale=scale)
    f_exp.add_scatter(x=df['theta'], y=df['pr'], mode='lines', name=f'{angle}°')
    f2_exp.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', name=f'{angle}°')
    exp_dfs.append(df)

for hs in np.arange(120, 164, 4):
    df = extract_theory_df(I_exp, r_vals, theta_vals, sigma_large=hs, e_scale=scale)
    f_exp.add_scatter(x=df['theta'], y=df['pr'], mode='lines', name=f'hs={hs}')
    f2_exp.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', name=f'hs={hs}')
    exp_dfs.append(df)

for ls in np.arange(12, 40, 2):
    df = extract_theory_df(I_exp, r_vals, theta_vals, sigma_small=ls, e_scale=scale)
    f_exp.add_scatter(x=df['theta'], y=df['pr'], mode='lines', name=f'ls={ls}')
    f2_exp.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', name=f'ls={ls}')
    exp_dfs.append(df)
f_exp.show()
f2_exp.update_layout(
        title="Experimental Group Delay vs Energy",
        yaxis_title="Group Delay",
        xaxis_title="Energy (E)",
        yaxis_range=[0, 250],
        xaxis_range=[0.015, 0.035],
).show()

In [ ]:
combined_fig = go.Figure().update_layout(
        title="Group Delay vs Energy (Combined Theory and Experiment)",
        yaxis_title="Group Delay",
        xaxis_title="Energy (E)",
        yaxis_range=[0, 200],
        xaxis_range=[0.015, 0.035],
        legend=dict(x=0.01, y=1, traceorder='normal', orientation='h')
)

for i, df in enumerate(theory_dfs):
    combined_fig.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', line_color='red', line_width=1,
                             name='Theory', showlegend=(i == 0))
for i, df in enumerate(exp_dfs):
    combined_fig.add_scatter(x=df['E'], y=df['group_delay'], mode='lines', line_color='blue', line_width=1,
                             name='Experiment', showlegend=(i == 0))

combined_fig.show()

In [ ]:
combined_fig_2 = go.Figure().update_layout(
        title="Phase Delay (Combined Theory and Experiment)",
        yaxis_title="Phase Delay",
        xaxis_title="Energy (E)",
        yaxis_range=[0, 200],
        xaxis_range=[0.015, 0.035],
        legend=dict(x=0.01, y=1, traceorder='normal', orientation='h')
)

for i, df in enumerate(theory_dfs):
    combined_fig_2.add_scatter(x=df['E'], y=2 * df['theta'] / df['E'], mode='lines', line_color='red', line_width=1,
                               name='Theory', showlegend=(i == 0))
for i, df in enumerate(exp_dfs):
    combined_fig_2.add_scatter(x=df['E'], y=2 * (df['theta'] - 2.62941 + np.pi) / df['E'], mode='lines',
                               line_color='blue', line_width=1, name='Experiment', showlegend=(i == 0))
combined_fig_2.show()

In [ ]:
def resample_and_plot(theory_dfs, exp_dfs, num_points=200, angular_shift=np.pi - 2.62941):
    # assemble full E-range
    all_E = np.hstack([df['E'].values for df in theory_dfs + exp_dfs])
    E_min, E_max = all_E.min(), all_E.max()
    E_grid = np.linspace(E_min, E_max, num_points)

    # convert atomic units → eV/fs
    ENERGY_CONV = 27.2114  # 1 a.u. energy = 27.2114 eV
    TIME_CONV_FS = 0.02418884  # 1 a.u. time   = 0.02418884 fs
    E_ev_grid = E_grid * ENERGY_CONV

    def compute_stats(dfs):
        gd = np.vstack([np.interp(E_grid, df['E'], df['group_delay']) for df in dfs])
        th = np.vstack([np.interp(E_grid, df['E'], df['theta']) for df in dfs])
        sg = np.vstack([np.interp(E_grid, df['E'], df['signal']) for df in dfs])
        return {
            'gd_mean': gd.mean(axis=0), 'gd_std': gd.std(axis=0),
            'th_mean': th.mean(axis=0), 'th_std': th.std(axis=0),
            'sg_mean': sg.mean(axis=0), 'sg_std': sg.std(axis=0),
        }

    theory = compute_stats(theory_dfs)
    experi = compute_stats(exp_dfs)

    # Group delay (fs) vs Energy (eV)
    fig1 = go.Figure()
    # Theory band
    fig1.add_trace(go.Scatter(
            x=E_ev_grid, y=(theory['gd_mean'] + theory['gd_std']) * TIME_CONV_FS,
            line=dict(color='blue', width=0), showlegend=False, hoverinfo='skip'
    ))
    fig1.add_trace(go.Scatter(
            x=E_ev_grid, y=(theory['gd_mean'] - theory['gd_std']) * TIME_CONV_FS,
            fill='tonexty', fillcolor='rgba(0,0,255,0.2)',
            line=dict(color='blue', width=0), showlegend=False, hoverinfo='skip'
    ))
    # Theory mean
    fig1.add_trace(go.Scatter(
            x=E_ev_grid, y=theory['gd_mean'] * TIME_CONV_FS,
            mode='lines', name='Theory', line_color='blue'
    ))
    # Experiment band
    fig1.add_trace(go.Scatter(
            x=E_ev_grid, y=(experi['gd_mean'] + experi['gd_std']) * TIME_CONV_FS,
            line=dict(color='red', width=0), showlegend=False, hoverinfo='skip'
    ))
    fig1.add_trace(go.Scatter(
            x=E_ev_grid, y=(experi['gd_mean'] - experi['gd_std']) * TIME_CONV_FS,
            fill='tonexty', fillcolor='rgba(255,0,0,0.2)',
            line=dict(color='red', width=0), showlegend=False, hoverinfo='skip'
    ))
    # Experiment mean
    fig1.add_trace(go.Scatter(
            x=E_ev_grid, y=experi['gd_mean'] * TIME_CONV_FS,
            mode='lines', name='Experiment', line_color='red'
    ))
    fig1.update_layout(
            title="Group Delay vs Energy (Combined Theory and Experiment)",
            xaxis_title="Energy (eV)", yaxis_title="Group Delay (fs)",
            xaxis_range=[0.3, 0.9],
            yaxis_range=[0, 5],
            legend=dict(x=0.01, y=1, orientation='h')
    )

    fig1.add_hline(y=2.67, line_color='black', line_dash='dash',
                   annotation_text="Laser Period", annotation_position="top left")
    fig1.show()

    # Phase (degrees) vs Energy (eV)
    fig2 = go.Figure()
    # Theory band
    fig2.add_trace(go.Scatter(
            x=E_ev_grid, y=np.degrees(theory['th_mean'] + theory['th_std']),
            line=dict(color='blue', width=0), showlegend=False, hoverinfo='skip'
    ))
    fig2.add_trace(go.Scatter(
            x=E_ev_grid, y=np.degrees(theory['th_mean'] - theory['th_std']),
            fill='tonexty', fillcolor='rgba(0,0,255,0.2)',
            line=dict(color='blue', width=0), showlegend=False, hoverinfo='skip'
    ))
    # Theory mean
    fig2.add_trace(go.Scatter(
            x=E_ev_grid, y=np.degrees(theory['th_mean']),
            mode='lines', name='Theory', line_color='blue'
    ))
    # Experiment band
    exp_phase = experi['th_mean'] + angular_shift
    fig2.add_trace(go.Scatter(
            x=E_ev_grid, y=np.degrees(exp_phase + experi['th_std']),
            line=dict(color='red', width=0), showlegend=False, hoverinfo='skip'
    ))
    fig2.add_trace(go.Scatter(
            x=E_ev_grid, y=np.degrees(exp_phase - experi['th_std']),
            fill='tonexty', fillcolor='rgba(255,0,0,0.2)',
            line=dict(color='red', width=0), showlegend=False, hoverinfo='skip'
    ))
    # Experiment mean
    fig2.add_trace(go.Scatter(
            x=E_ev_grid, y=np.degrees(exp_phase),
            mode='lines', name='Experiment', line_color='red'
    ))
    fig2.update_layout(
            title="Phase vs Energy (Combined Theory and Experiment)",
            xaxis_title="Energy (eV)", yaxis_title="Phase (degrees)",
            xaxis_range=[0.3, 0.9],
            yaxis_range=[0, 135],
            legend=dict(x=0.01, y=1, orientation='h')
    )
    fig2.show()

    # Phase delay (fs) vs Energy (eV)
    fig3 = go.Figure()
    # Theory band
    fig3.add_trace(go.Scatter(
            x=E_ev_grid, y=2 * (theory['th_mean'] + theory['th_std'] + np.pi / 2) / E_grid * TIME_CONV_FS,
            line=dict(color='blue', width=0), showlegend=False, hoverinfo='skip'
    ))
    fig3.add_trace(go.Scatter(
            x=E_ev_grid, y=2 * (theory['th_mean'] - theory['th_std'] + np.pi / 2) / E_grid * TIME_CONV_FS,
            fill='tonexty', fillcolor='rgba(0,0,255,0.2)',
            line=dict(color='blue', width=0), showlegend=False, hoverinfo='skip'
    ))
    # Theory mean
    fig3.add_trace(go.Scatter(
            x=E_ev_grid, y=2 * (theory['th_mean'] + np.pi / 2) / E_grid * TIME_CONV_FS,
            mode='lines', name='Theory', line_color='blue'
    ))
    # Experiment band
    fig3.add_trace(go.Scatter(
            x=E_ev_grid, y=2 * (exp_phase + experi['th_std'] + np.pi / 2) / E_grid * TIME_CONV_FS,
            line=dict(color='red', width=0), showlegend=False, hoverinfo='skip'
    ))
    fig3.add_trace(go.Scatter(
            x=E_ev_grid, y=2 * (exp_phase - experi['th_std'] + np.pi / 2) / E_grid * TIME_CONV_FS,
            fill='tonexty', fillcolor='rgba(255,0,0,0.2)',
            line=dict(color='red', width=0), showlegend=False, hoverinfo='skip'
    ))
    # Experiment mean
    fig3.add_trace(go.Scatter(
            x=E_ev_grid, y=2 * (exp_phase + np.pi / 2) / E_grid * TIME_CONV_FS,
            mode='lines', name='Experiment', line_color='red'
    ))
    fig3.update_layout(
            title="Phase Delay vs Energy (Combined Theory and Experiment)",
            xaxis_title="Energy (eV)", yaxis_title="Phase Delay (fs)",
            xaxis_range=[0.3, 0.9],
            yaxis_range=[5, 10],
            legend=dict(x=0.01, y=1, orientation='h')
    )
    fig3.show()

    return E_ev_grid, E_grid, exp_phase, experi, theory


# Resample and plot the combined data
d = resample_and_plot(theory_dfs, exp_dfs, num_points=100, angular_shift=0.6035 - np.radians(10))
E_ev_grid, E_grid, exp_phase, experi, theory = d
del d

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib import colors

TIME_CONV_FS = 0.02418884


def add_banded_line(fig, x, y, dy, color, showlegend=True, **kwargs):
    fig.add_scatter(
            x=x,
            y=y + dy,
            line=dict(width=0, color=color),
            showlegend=False,
            hoverinfo='skip',
            **kwargs,
    )
    fig.add_scatter(
            x=x,
            y=y - dy,
            line=dict(width=0, color=color),
            showlegend=False,
            hoverinfo='skip',
            fill='tonexty',
            fillcolor=f"rgba{colors.to_rgba(color, alpha=0.3)}",
            **kwargs,
    )
    fig.add_scatter(
            x=x,
            y=y,
            mode='lines',
            line=dict(width=2, color=color),
            marker=dict(color=color, size=5),
            showlegend=showlegend,
            **kwargs,
    )


fig = make_subplots(
        rows=2, cols=1,
        vertical_spacing=0.05,
        shared_xaxes=True,
).update_layout(
        height=800,
        width=600,
        margin=dict(t=0, r=0, b=0, l=0),
        template='simple_white+presentation',
        legend=dict(
                orientation="h",
                yanchor="top",
                y=1,
                xanchor="right",
                x=1,
        )
)

add_banded_line(fig, E_ev_grid, 2 * (theory['th_mean'] + np.pi / 2), 2 * theory['th_std'], 'blue', row=1, col=1,
                name='Theory')
add_banded_line(fig, E_ev_grid, 2 * (exp_phase + np.pi / 2), 2 * experi['th_std'], 'red', row=1, col=1,
                name='Experiment')

add_banded_line(
        fig,
        E_ev_grid,
        theory['gd_mean'] * TIME_CONV_FS,
        theory['gd_std'] * TIME_CONV_FS,
        'blue',
        row=2, col=1,
        name='Theory',
        showlegend=False,
)
add_banded_line(
        fig,
        E_ev_grid,
        experi['gd_mean'] * TIME_CONV_FS,
        experi['gd_std'] * TIME_CONV_FS,
        'red',
        row=2, col=1,
        name='Experiment',
        showlegend=False,
)

fig.update_xaxes(
        range=[0.4, 0.7],
        # minor_showgrid=True,
        ticks="inside",
        ticklen=6,
        mirror=True
)

fig.update_xaxes(
        row=2,
        title_text="Energy (eV)",
)

fig.update_yaxes(
        # minor_showgrid=True,
        ticks="inside",
        ticklen=6,
        mirror=True,
)

fig.update_yaxes(
        range=[4.5, 7],
        row=1,
        title_text="Phase Difference (rad)",
)
fig.update_yaxes(
        range=[0, 5],
        row=2,
        title_text="Group Delay (fs)",
        tickvals=[0, 1, 2, 3, 4, 5],
        # minor_tickvals=[0.25,0.5,0.75,1.25,1.5,1.75,2.25,2.5,2.75,3.25,3.5,3.75,4.25,4.5,4.75],
)

fig.add_hline(
        y=2.669,
        line_color='black',
        line_width=1,
        opacity=1,
        line_dash='dashdot',
        annotation_text="Laser Period",
        annotation_position="bottom right",
        row=2, col=1,
)

fig.add_annotation(
        text="(a)",
        xref="paper",
        yref="y domain",
        x=0.025,
        y=0.975,
        showarrow=False,
)
# fig.add_annotation(
#         text=r"$\tau_{p}=\hbar \frac{\varphi}{E}$",
#         xref="x",
#         yref="y",
#         x=0.6,
#         y=2.9,
#         showarrow=False,
# )

fig.add_annotation(
        text="(b)",
        xref="paper",
        yref="y2 domain",
        x=0.025,
        y=0.975,
        showarrow=False,
)
fig.show()
fig.write_image(
        r"C:\Users\mcman\DataspellProjects\vmi-analysis\Xenon Delay Figure.png",
        scale=5
)

In [ ]:
ls = 25
hs = 110

df = extract_theory_df(I_exp, r_vals, theta_vals, angle=45, e_scale=scale, sigma_small=ls, sigma_large=hs)

dog_fig = make_subplots(2, 1, shared_xaxes=True, vertical_spacing=0.05, )
dog_fig.update_layout(height=600, width=600, margin=dict(t=0, r=0, b=0, l=0),
                      template='simple_white+presentation', )
dog_fig.update_xaxes(title_text=r"$\phi~(rad)$", row=2)
dog_fig.update_yaxes(title_text=r"$p_r~(a.u.)$", )

dog_fig.add_trace(
        go.Heatmap(
                x=theta_vals, y=r_vals * scale, z=I_exp, showscale=False, colorscale=px.colors.sequential.Inferno,
        )
)
dog_fig.add_trace(
        go.Heatmap(
                x=theta_vals, y=r_vals * scale, z=skimage.filters.difference_of_gaussians(I_exp, ls, hs),
                showscale=False, colorscale=px.colors.sequential.Inferno,
        ),
        row=2, col=1,
)
dog_fig.add_scatter(x=df['theta'], y=df['pr'] * scale, mode='lines', line_color="white", showlegend=False)
dog_fig.add_scatter(x=df['theta'], y=df['pr'] * scale, mode='lines', line_color="white", row=2, col=1, showlegend=False)
dog_fig.add_annotation(
        text="(a)",
        xref="paper",
        yref="y domain",
        font_color="white",
        x=0.025,
        y=0.975,
        showarrow=False,
)
dog_fig.add_annotation(
        text="(b)",
        font_color="white",
        xref="paper",
        yref="y2 domain",
        x=0.025,
        y=0.975,
        showarrow=False,
)
dog_fig.write_image(
        r"C:\Users\mcman\DataspellProjects\vmi-analysis\Appendix Smoothing.png",
        scale=5
)
dog_fig.show()